# PyTorch 입문 — 텐서 · 선형회귀 · 경사하강

**PyTorch** = 숫자 덩어리(텐서)를 쌓아 딥러닝 모델을 만드는 도구입니다.  
이 노트북은 [`../session01.md`](../session01.md)에서 본 감각을 **바로 실행**해 보는 입문용입니다.

> **이 노트북이 하는 일**  
> 텐서를 만들고, 미분을 한 번 돌려 본 뒤, `y = wx + b`를 손으로·모듈로 각각 학습합니다.

환경: `torch-env` 커널을 쓰는 것을 권장합니다. (`session01` 9절 참고)


## 1. 준비 — import · 버전 · device

**device** = 계산을 어디서 할지입니다. 이 입문에서는 **CPU**만 씁니다.


In [ ]:
import torch

# 설치·환경이 맞는지 버전으로 먼저 확인합니다
print("torch 버전:", torch.__version__)

# GPU가 있어도 입문 실습은 CPU로 통일합니다
device = torch.device("cpu")
print("device:", device)


torch 버전: 2.5.1
device: cpu


## 2. 텐서 만들기 · shape · NumPy 변환

**텐서(Tensor)** = NumPy 배열과 비슷한 **숫자 상자**입니다.  
**shape** = 상자가 몇 줄·몇 칸인지입니다.


In [ ]:
# 1차원 텐서 = 숫자 한 줄
x = torch.tensor([1.0, 2.0, 3.0, 4.0]) # 1차원 tensor 생성
# 1차원 tensor 출력
print("x:", x) # x: tensor([1., 2., 3., 4.])
# 1차원 tensor 의 shape 출력
print("shape:", x.shape) #shape: torch.Size([4])
# 1차원 tensor 의 dtype 출력
print("dtype:", x.dtype) #dtype: torch.float32

# 2줄 표(2차원)도 같은 방식으로 만듭니다
mat = torch.tensor([[1.0, 2.0], [3.0, 4.0]]) # 2차원 tensor 생성
# 2차원 tensor 의 shape 출력
print("mat shape:", mat.shape) # mat shape: torch.Size([2, 2])

# NumPy로 갔다가 다시 텐서로 — 서로 왕복할 수 있습니다
x_np = x.numpy() # tensor 를 numpy 로 변환
x_back = torch.from_numpy(x_np) # numpy 를 tensor 로 변환

# numpy 로 변환한 tensor 출력
print("NumPy:", x_np) # NumPy: [1. 2. 3. 4.]
# tensor 로 변환한 numpy 출력
print("다시 텐서:", x_back) # 다시 텐서: tensor([1., 2., 3., 4.])


x: tensor([1., 2., 3., 4.])
shape: torch.Size([4])
dtype: torch.float32
mat shape: torch.Size([2, 2])
NumPy: [1. 2. 3. 4.]
다시 텐서: tensor([1., 2., 3., 4.])


## 3. 연산 · requires_grad · backward

**requires_grad=True** = "이 값으로 미분할 거니 계산 과정을 기억해 주세요"라는 표시입니다.  
**backward** = 기억해 둔 계산을 거꾸로 따라가며 **기울기(gradient)** 를 구하는 호출입니다.


In [ ]:
# w를 학습할 후보로 두고, 미분 추적을 켭니다

# **requires_grad=True** 는 미분 추적을 켜는 것
# 미분 추적이란 미분을 할 때 계산 과정을 기억해 두는 것
# 미분 추적을 켜면 미분을 할 때 계산 과정을 기억해 두기 때문에 미분을 할 때 계산 과정을 기억해 두는 것
w = torch.tensor(2.0, requires_grad=True)
# w를 학습할 후보로 두고, 미분 추적을 켭니다

# 간단한 식: loss = (w * 3 - 4)^2  → w=2일 때 값을 계산
y = w * 3.0
loss = (y - 4.0) ** 2 # ** 는 제곱을 의미합니다
# loss 출력
print("loss:", loss.item()) # loss: 4.0

# 한 번만 backward — w.grad에 d(loss)/d(w)가 쌓입니다
# backward 는 미분을 할 때 계산 과정을 기억해 두는 것
# backward를 통해 .grad(기울기)가 생긴다
loss.backward()

# w.grad 는 미분을 할 때 계산 과정을 기억해 두는 것
print("w.grad:", w.grad) # w.grad: tensor([12.])

# 식은 PyTorch가 기억한다
# loss.backward() → 각 파라미터에 .grad(기울기)가 생긴다
# 그 grad로 값을 고친다 (빼기 or optimizer.step)

loss: 4.0
w.grad: tensor(12.)


## 4. 수동 경사하강 — `y = wx + b`

**경사하강** = 오차(손실)가 작아지도록 `w`, `b`를 조금씩 고치는 반복입니다.  
**MSE** = 예측과 정답 차이의 제곱 평균(평균 제곱 오차)입니다.

정답은 `y = 2x + 1`로 두고, 모델이 `w`, `b`를 스스로 맞추게 합니다.


In [ ]:
torch.manual_seed(0) # manual_seed는 랜덤 시드를 설정하는 것

# 입력 x와 정답 y (진짜 식: y = 2x + 1)
# 열 벡터로 만들기
x = torch.linspace(-1.0, 1.0, 20).unsqueeze(1)
# linspace는 일정한 간격으로 숫자를 생성하는 함수 (-1.0부터 1.0까지 20개의 숫자를 일정한 간격으로 생성)
# unsqueeze(1)은 1차원 텐서를 2차원 텐서로 변환하는 것

# torch.linspace(-1.0, 1.0, 20)
# tensor([-1., -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]) 값이 생성
# .unsqueeze(1) 를 통해,
# tensor([[-1.], [-0.8], [-0.6], [-0.4], [-0.2], [0.0], [0.2], [0.4], [0.6], [0.8], [1.0]]) 형태로 변환

y_true = 2.0 * x + 1.0

# 시작할 때 w, b는 아무 값이나 — 학습으로 맞춥니다
# randn는 랜덤 값을 생성하는 함수 (1, ) 은 1차원 텐서를 생성하는 것
# requires_grad=True는 미분 추적을 켜는 것
w = torch.randn(1, requires_grad=True) # (1, 1) 형태의 랜덤 값을 생성
b = torch.randn(1, requires_grad=True) # (1, 1) 형태의 랜덤 값을 생성
lr = 0.1  # 학습률 = 한 번에 얼마나 고칠지

for step in range(80): # 80번 반복
    y_pred = w * x + b
    # MSE = 차이의 제곱을 평균
    # mean()은 평균을 계산하는 함수
    loss = ((y_pred - y_true) ** 2).mean()

    # 이전 기울기를 지운 뒤 새로 계산합니다
    # w.grad가 None이 아닌지 확인
    if w.grad is not None:
        # w.grad.zero_()는 w.grad를 0으로 초기화하는 것
        w.grad.zero_()
    # b.grad가 None이 아닌지 확인
    if b.grad is not None:
        # b.grad.zero_()는 b.grad를 0으로 초기화하는 것
        b.grad.zero_()

    loss.backward()

    # 추적 밖에서 값만 갱신 (이 줄 자체는 미분 대상이 아님)
    # with torch.no_grad(): 는 미분 추적을 끄는 것
    with torch.no_grad():
        # w -= lr * w.grad 는 w를 lr만큼 갱신하는 것
        w -= lr * w.grad
        # b -= lr * b.grad 는 b를 lr만큼 갱신하는 것
        b -= lr * b.grad
    # 갱신 하는 이유는 손실을 줄이기 위해서

    if step % 20 == 0 or step == 79:
        print(f"step {step:3d} | loss={loss.item():.6f} | w={w.item():.3f} | b={b.item():.3f}")

# 랜덤 시드를 설정했기 때문에
# w≈2 로 w 는 2에 가까워지고,
# b≈1 로 b 는 1에 가까워지는 것을 확인할 수 있습니다.
print("목표: w≈2, b≈1")

# 수동 경사하강을 통해 w≈2, b≈1 로 근사값을 찾는 것을 확인할 수 있습니다.

# 직선의 방정식은 y = wx + b 이기 때문에 w, b 두개를 학습해야 한다.
# 예를 들어, y = 2x + 1 이라는 직선의 방정식이 있을 때,
# w = 2, b = 1 이라는 값을 찾는 것이다.
# 이때, w와 b 두개를 학습해야 하는 이유는 직선의 방정식을 만족시키기 위해서는 w와 b 두개를 학습해야 하는 것이다.
# 즉, 직선의 방정식을 만족시키기 위해서는 w와 b 두개를 학습해야 하는 것이다.


step   0 | loss=1.750579 | w=1.575 | b=-0.035
step  20 | loss=0.003856 | w=1.908 | b=0.988
step  40 | loss=0.000170 | w=1.980 | b=1.000
step  60 | loss=0.000008 | w=1.996 | b=1.000
step  79 | loss=0.000000 | w=1.999 | b=1.000
목표: w≈2, b≈1


## 5. (선택) `nn.Linear` + `optim.SGD`

같은 문제를 **모듈**과 **옵티마이저**로 풀면, 손으로 `w`/`b`를 갱신하지 않아도 됩니다.  
`nn.Linear(1, 1)` = 입력 1개 → 출력 1개인 직선 `y = wx + b`입니다.


In [ ]:
import torch.nn as nn
import torch.optim as optim

# manual_seed는 랜덤 시드를 설정하는 것
# 랜덤 시드를 설정하면 랜덤 값을 고정시킬 수 있다.
torch.manual_seed(1)

x = torch.linspace(-1.0, 1.0, 20).unsqueeze(1)
# linspace는 일정한 간격으로 숫자를 생성하는 함수 (-1.0부터 1.0까지 20개의 숫자를 일정한 간격으로 생성)
# unsqueeze(1)은 1차원 텐서를 2차원 텐서로 변환하는 것
# torch.linspace(-1.0, 1.0, 20)
# tensor([-1., -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]) 값이 생성
# .unsqueeze(1) 를 통해,
# tensor([[-1.], [-0.8], [-0.6], [-0.4], [-0.2], [0.0], [0.2], [0.4], [0.6], [0.8], [1.0]]) 형태로 변환
y_true = 2.0 * x + 1.0

# nn.Linear(1, 1) 는 입력 1개 → 출력 1개인 직선 y = wx + b 를 만드는 것
model = nn.Linear(1, 1)  # 내부적으로 w, b를 가짐
# nn.MSELoss() 는 손실 = MSE(평균 제곱 오차)를 계산하는 것 - 예측과 정답이 얼마나 차이가 나는지 계산하는 것
criterion = nn.MSELoss()  # MSE Loss 를 계산하는 것. why? 오차를 줄이기 위해서
optimizer = optim.SGD(model.parameters(), lr=0.1) # 학습률 = 한 번에 얼마나 고칠지
# SGD는 경사하강법 도구를 준비하는 것
# 경사하강법은 손실을 줄이는 방향으로 파라미터를 고치는 것을 의미
# 아직 값을 바꾸지는 않고, 옵티마이저 만 준비

for step in range(80):
    y_pred = model(x)
    # criterion(y_pred, y_true)는 예측과 정답이 얼마나 차이가 나는지 - 오차를 계산하는 것
    loss = criterion(y_pred, y_true)

    optimizer.zero_grad()  # 이전 기울기 비우기
    loss.backward()
    optimizer.step()  # w, b 갱신
    # step 함수는 경사하강법을 실행
    # 실제로 w, b를 한 번 고치는 줄임

    if step % 20 == 0 or step == 79:
        # model.weight.item()는 w의 값을 가져오는 것
        w = model.weight.item()
        # model.bias.item()는 b의 값을 가져오는 것
        b = model.bias.item()
        print(f"step {step:3d} | loss={loss.item():.6f} | w={w:.3f} | b={b:.3f}")

print("목표: w≈2, b≈1")


step   0 | loss=2.889735 | w=0.625 | b=-0.153
step  20 | loss=0.038296 | w=1.702 | b=0.987
step  40 | loss=0.001780 | w=1.936 | b=1.000
step  60 | loss=0.000083 | w=1.986 | b=1.000
step  79 | loss=0.000005 | w=1.997 | b=1.000
목표: w≈2, b≈1


## 정리

1. 텐서는 NumPy처럼 생긴 숫자 상자입니다.
2. `requires_grad` + `backward`로 기울기를 구합니다.
3. 경사하강은 "손실을 줄이는 방향"으로 `w`, `b`를 고칩니다.
4. `nn.Linear` + `optim.SGD`는 같은 일을 모듈이 대신합니다.

이론 복습: [`../session01.md`](../session01.md)
